In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Cubic calibration coefficients from thermocouple_calibration.ipynb
# V_mV = a3*t^3 + a2*t^2 + a1*t + a0,  t = (T_K - T_MEAN) / T_STD
CAL_COEFFS = np.array([-5.371838e+01, 1.037465e+02, 1961.945128, 1227.4515])
CAL_T_MEAN = 398.27    # K
CAL_T_STD  = 193.13    # K
CAL_T_MIN_K = 73.15    # K  (-200 °C)
CAL_T_MAX_K = 573.15   # K  (300 °C)

def voltage_to_temperature(voltage_V):
    """Convert thermocouple voltage (V) to temperature (°C) via cubic calibration."""
    voltage_mV = voltage_V * 1000.0
    shifted = CAL_COEFFS.copy()
    shifted[-1] -= voltage_mV

    roots = np.roots(shifted)
    real_roots = roots[np.abs(roots.imag) < 1e-6].real
    T_candidates = real_roots * CAL_T_STD + CAL_T_MEAN
    in_domain = T_candidates[(T_candidates >= CAL_T_MIN_K) & (T_candidates <= CAL_T_MAX_K)]

    if len(in_domain) == 0:
        return np.nan
    return float(in_domain[0]) - 273.15

df = pd.read_csv('nikitas_experimets.csv')
df['temperature_C'] = df['thermo_volt'].apply(voltage_to_temperature)
df['capacitance_pF'] = df['capacitance_value'] * 1e12

# Filter out outliers where capacitance > 100 pF (1e-10 F)
df = df[df['capacitance_value'] <= 1e-10]

# Filter out points below -30 °C
df = df[df['temperature_C'] >= -30]
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['temperature_C'], df['capacitance_pF'], s=10, alpha=0.6, label='Data')

# Linear best fit
mask = df['temperature_C'].notna() & df['capacitance_pF'].notna()
m, b = np.polyfit(df.loc[mask, 'temperature_C'], df.loc[mask, 'capacitance_pF'], 1)
T_fit = np.linspace(df['temperature_C'].min(), df['temperature_C'].max(), 200)
ax.plot(T_fit, m * T_fit + b, 'r-', linewidth=2, label=f'Linear fit: C = {m:.4f}T + {b:.2f} pF')

ax.set_xlabel('Temperature (°C)', fontsize=13)
ax.set_ylabel('Capacitance (pF)', fontsize=13)
ax.set_title('Capacitance vs Temperature', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()